# SACENet: Sarcasm-Aware Explainable Early Detection of Cyberbullying
## IEEE Research Paper | Kaggle Edition (GitHub → Kaggle)

**Model**: MuRIL (google/muril-base-cased) + LLM Sarcasm Engine  
**Dataset**: MC-Hinglish1.0  
**XAI**: SHAP Token-Level Attribution  
**GitHub**: https://github.com/aishwaryakg2415/cyberbullying-detection

---
### Setup:
1. Add MC-Hinglish1.0 dataset via **+ Add Data** on the right panel
2. Enable **GPU T4** in Settings → Accelerator
3. Click **Run All**
---

## Step 1: Clone GitHub Repo & Install

In [ ]:
import os, sys

GITHUB_REPO = 'https://github.com/aishwaryakg2415/cyberbullying-detection.git'
REPO_DIR    = '/kaggle/working/cyberbullying-detection'

# Clone repo
if not os.path.exists(REPO_DIR):
    os.system(f'git clone {GITHUB_REPO} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')

# Add repo to Python path so we can import modules
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('Repo ready at:', REPO_DIR)
print('Files:', os.listdir(REPO_DIR))

In [ ]:
# Install extra packages (torch, transformers pre-installed on Kaggle)
os.system('pip install shap wordcloud evaluate -q')
print('Packages ready.')

## Step 2: Configure Dataset Paths

In [ ]:
import glob

# Find your dataset (added via + Add Data)
# Kaggle mounts datasets at /kaggle/input/<dataset-slug>/
KAGGLE_INPUT = '/kaggle/input'
all_csvs = glob.glob(f'{KAGGLE_INPUT}/**/*.csv', recursive=True)
print('All CSVs found:')
for f in all_csvs:
    print(f'  {f}')

# Auto-detect paths
TRAIN_PATH = next((f for f in all_csvs if 'train' in f.lower() and 'hinglish' in f.lower().replace('-','').replace('_','')), 
             next((f for f in all_csvs if 'train' in f.lower()), None))
TEST_PATH  = next((f for f in all_csvs if 'test'  in f.lower() and 'hinglish' in f.lower().replace('-','').replace('_','')), 
             next((f for f in all_csvs if 'test'  in f.lower()), None))
HUMAN_PATH = next((f for f in all_csvs if 'human' in f.lower()), None)
AI_PATH    = next((f for f in all_csvs if 'ai_lab' in f.lower() or ('ai' in f.lower() and 'label' in f.lower())), None)

print(f'\nTrain : {TRAIN_PATH}')
print(f'Test  : {TEST_PATH}')
print(f'Human : {HUMAN_PATH}')
print(f'AI    : {AI_PATH}')

assert TRAIN_PATH and TEST_PATH, 'Could not find MC-Hinglish CSVs. Please check + Add Data.'

## Step 3: Override config.py paths & Load Everything

In [ ]:
# Override DATA_DIR in config before importing modules
import importlib
import config
config.DATA_DIR       = os.path.dirname(TRAIN_PATH)
config.MODELS_DIR     = '/kaggle/working/saved_model'
config.RESULTS_DIR    = '/kaggle/working/results'
config.PLOTS_DIR      = '/kaggle/working/plots'
os.makedirs(config.MODELS_DIR,  exist_ok=True)
os.makedirs(config.RESULTS_DIR, exist_ok=True)
os.makedirs(config.PLOTS_DIR,   exist_ok=True)

# Set correct file names
import dataset as ds_module
ds_module.MC_TRAIN_PATH  = TRAIN_PATH
ds_module.MC_TEST_PATH   = TEST_PATH
ds_module.HUMAN_SAR_PATH = HUMAN_PATH or ''
ds_module.AI_SAR_PATH    = AI_PATH    or ''

from dataset import CyberbullyingDataset, TextPreprocessor
from sarcasm_engine import SarcasmEngine
from model import SACENet

print('All modules imported from GitHub repo.')

## Step 4: Load Dataset

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

print('Loading MC-Hinglish1.0 dataset...')
cyberbullying_ds = CyberbullyingDataset(balance=True)
stats = cyberbullying_ds.get_stats()
print(json.dumps(stats, indent=2))

train_df, val_df, test_df = cyberbullying_ds.get_splits()
print(f'\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Dataset visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

train_df['label_str'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B3','#937860','#DA8BC3'],
    edgecolor='white')
axes[0].set_title('MC-Hinglish1.0 — Label Distribution', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

train_df['label'].value_counts().rename({0:'Non-Bully',1:'Cyberbullying'}).plot(
    kind='pie', ax=axes[1], autopct='%1.1f%%',
    colors=['#4C72B0','#C44E52'], startangle=90)
axes[1].set_title('Binary Distribution', fontweight='bold')
axes[1].set_ylabel('')

train_df.groupby('label_str')['sarcastic'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=axes[2], color='#DD8452', edgecolor='white')
axes[2].set_title('Sarcasm Rate by Category', fontweight='bold')
axes[2].tick_params(axis='x', rotation=30)
axes[2].set_ylabel('Proportion Sarcastic')

plt.tight_layout()
plt.savefig('/kaggle/working/plots/dataset_stats.png', dpi=200)
plt.show()

## Step 5: Sarcasm Engine Demo

In [ ]:
# Using local model (no API key needed on Kaggle)
sarcasm = SarcasmEngine(provider='local', use_cache=True)

hinglish_examples = [
    ('Wah wah! Kya genius hai tu. Tabhi toh fail ho gaya phir se.',  'Sarcastic bully'),
    ('Tu bilkul bekaar hai, teri aukaat kya hai bhai.',               'Direct bully'),
    ('Bhai tera project ekdum mast tha, proud feel kiya.',            'Non-bully'),
    ('Haan haan BAHUT talented hai tu. Tabhi koi nahi sunta.',        'Sarcastic bully'),
    ('Aaj kya plan hai? Movie dekhne chalein kya?',                   'Normal chat'),
    ('Badhai ho genius ji! Itni mushkil cheez finally samajh aayi.',  'Sarcastic bully'),
]

rows = []
print('SARCASM ENGINE RESULTS (Hinglish)\n' + '='*60)
for text, cat in hinglish_examples:
    r = sarcasm.detect(text)
    icon = 'SARCASTIC' if r.is_sarcastic else 'SINCERE  '
    print(f'{icon} [{cat}]')
    print(f'  Text  : {text[:65]}')
    print(f'  Score : {r.sarcasm_score:.3f} ({r.confidence}) | Cues: {r.indicators or "None"}')
    rows.append({'Category':cat,'Text':text[:40]+'...','Score':r.sarcasm_score,
                 'Sarcastic':r.is_sarcastic,'Confidence':r.confidence})
    print()

display(pd.DataFrame(rows))

## Step 6: Fine-Tune MuRIL

In [ ]:
from train import SACENetTrainer

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

print('Tokenising dataset for MuRIL...')
hf_dataset = cyberbullying_ds.get_hf_dataset(tokenizer_name=config.CYBERBULLYING_MODEL)

# Override training config for Kaggle GPU
kaggle_config = {
    **config.TRAINING_CONFIG,
    'output_dir'                   : '/kaggle/working/saved_model',
    'per_device_train_batch_size'  : 32,
    'per_device_eval_batch_size'   : 64,
    'num_train_epochs'             : 5,
    'fp16'                         : torch.cuda.is_available(),
    'logging_steps'                : 50,
}

trainer = SACENetTrainer(
    base_model=config.CYBERBULLYING_MODEL,
    output_dir='/kaggle/working/saved_model',
    config=kaggle_config,
)

print('Starting MuRIL fine-tuning on MC-Hinglish1.0...')
train_results = trainer.train(hf_dataset)
print('\nTraining complete!')
print('Test metrics:', json.dumps(train_results['test_metrics'], indent=2))

## Step 7: Full SACENet Pipeline + Evaluation

In [ ]:
from evaluate import run_full_evaluation

# Override paths for Kaggle
config.RESULTS_DIR = '/kaggle/working/results'
config.PLOTS_DIR   = '/kaggle/working/plots'

print('Running full evaluation...')
eval_results = run_full_evaluation(
    model_path='/kaggle/working/saved_model',
    save_results=True
)

print('\nMODEL COMPARISON:')
comp_df = pd.DataFrame(eval_results['model_comparison'])
display(comp_df.set_index('model').round(4))

## Step 8: SHAP Explainability

In [ ]:
# Load the fine-tuned SACENet
model = SACENet(
    finetuned_path='/kaggle/working/saved_model',
    sarcasm_provider='local',
    enable_shap=True,
)

# Explain sarcastic cyberbullying examples
examples = [
    ('Wah wah! Kya genius hai tu. Tabhi toh fail ho gaya phir se, badhai ho.', 'twitter'),
    ('Tu bilkul bekaar hai yaar, teri aukaat kya hai.',                         'instagram'),
    ('Bhai tera kaam ekdum mast tha, sach mein proud feel kiya.',               'twitter'),
]

print('='*70)
print('  SACENet — HINGLISH DETECTION WITH SHAP')
print('='*70)
for text, platform in examples:
    result = model.detect(text, platform=platform, explain=True)
    label  = 'CYBERBULLYING' if result.is_cyberbullying else 'NON-BULLYING'
    print(f'\n[{platform.upper()}] {text[:65]}')
    print(f'  Prediction : {label} (prob={result.cyberbullying_prob:.3f}, {result.confidence})')
    print(f'  Base prob  : {result.base_cyberbullying_prob:.3f}')
    if result.sarcasm_result:
        s = result.sarcasm_result
        print(f'  Sarcasm    : {s.sarcasm_score:.3f} | Cues: {s.indicators or "None"}')
    if result.sarcasm_was_applied:
        print(f'  Boost      : +{result.sarcasm_boost_applied:.3f} applied')
    if result.explanation:
        print(f'  SHAP       : {result.explanation.shap_narrative[:120]}')

## Step 9: Early Detection Demo

In [ ]:
import numpy as np

conversation = [
    'Bhai teri nayi photo dekhi.',
    'Wah wah, bahut himmat hai tune yeh post karne ki.',
    'Seriously ITNA confidence kahan se aata hai tujhe.',
    'Most log toh mirror dekh ke rote hain par tu selfie leta hai.',
    'Haan haan, BAHUT popular hai tu. Tabhi akele baith ke post karta hai.',
]

print('Hinglish Sarcastic Escalation Conversation (Early Detection):')
print('='*65)
for i, msg in enumerate(conversation, 1):
    print(f'  [{i}] {msg}')

early = model.early_detect_conversation(
    messages=conversation, platform='instagram', k=3, explain=False
)

print(f"\nRisk Trajectory : {[round(r,3) for r in early['risk_trajectory']]}")
print(f"Escalation      : {early['escalation_detected']}")
print(f"Final Alert     : {'CYBERBULLYING DETECTED' if early['early_result'].is_cyberbullying else 'Safe'}")

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
k_vals = list(range(1, len(early['risk_trajectory'])+1))
ax.plot(k_vals, early['risk_trajectory'], 'o-', color='#C44E52', lw=2.5, ms=10)
ax.fill_between(k_vals, early['risk_trajectory'], alpha=0.15, color='#C44E52')
ax.axhline(0.55, color='black', ls='--', lw=1.5, label='Decision Threshold (0.55)')
ax.set_xlabel('Message Number (K)', fontsize=12)
ax.set_ylabel('Cyberbullying Risk Score', fontsize=12)
ax.set_title('Early Detection — Hinglish Sarcastic Escalation', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('/kaggle/working/plots/early_detection.png', dpi=200)
plt.show()

## Step 10: LaTeX Table + Download Outputs

In [ ]:
import os, glob

# LaTeX table
print('LaTeX Results Table:')
print('\\begin{table}[h]')
print('\\centering')
print('\\caption{Cyberbullying Detection on MC-Hinglish1.0}')
print('\\label{tab:results}')
print('\\begin{tabular}{lccccc}')
print('\\hline')
print('Model & Accuracy & F1 & Precision & Recall & ROC-AUC \\\\')
print('\\hline')
for m in eval_results['model_comparison']:
    auc = m.get('roc_auc', '-')
    auc_str = f"{auc:.4f}" if isinstance(auc, float) else auc
    print(f"{m['model']} & {m['accuracy']:.4f} & {m['f1_binary']:.4f} & {m['precision']:.4f} & {m['recall']:.4f} & {auc_str} \\\\")
print('\\hline')
print('\\end{tabular}')
print('\\end{table}')

print('\nGenerated output files:')
for f in glob.glob('/kaggle/working/**/*', recursive=True):
    if os.path.isfile(f):
        size = os.path.getsize(f) // 1024
        print(f'  {f}  [{size} KB]')